# Predict the rows that have no label yet

Learn how to predict the customers whose label is missing with static queries, and how to control which rows are scored.
This Kumo Engine module trains three small models on synthetic retail data (binary, multiclass and regression), reads each output layout, supplies your own population and covers the two rules that surprise people.

Use a Linux notebook prepared with the [Kumo Engine setup](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/linux-machine-deployment). A GPU is recommended for the FAST training preset; set `SMOKE_TEST = True` below for a quick check on a CPU-only machine.
Expect under a minute on a single L4 GPU after installation.

See [Predictive Queries](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/predictive-queries) for query syntax and [Making predictions](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/making-predictions#select-the-population) for populations.

## Install

Install Kumo Engine in this notebook kernel.

In [1]:
%pip install kumo-engine

## Choose a workspace

Set `KUMO_LOCAL_SSD` to a local SSD mount for faster scratch storage; otherwise the system temporary directory is used.
This cell creates a new directory on each run, with separate source and scratch directories, and does not replace existing data.
The last lines print the installed Kumo Engine version and the workspace.

In [2]:
import importlib.metadata
import os
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from kumoapi.model_plan import RunMode
from kumoapi.typing import Stype

import kumo.engine as kumo

workspace_root = Path(os.environ.get("KUMO_LOCAL_SSD", tempfile.gettempdir()))
workspace = Path(tempfile.mkdtemp(prefix="kumo-static-predictions-", dir=workspace_root))
source_dir = workspace / "source"
source_dir.mkdir()
kumo.configure(scratch_dir=workspace / "scratch")
print("Kumo Engine:", importlib.metadata.version("kumo-engine"))
print("Workspace:", workspace)

Kumo Engine: 1.0.1
Workspace: /tmp/kumo-static-predictions-96akbhma


## Set the training budget

Models train with the [FAST preset](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/model-plan-guide).
Set `SMOKE_TEST = True` to cap training at two short epochs for a quick check on a CPU-only machine; its scores carry no meaning.

In [3]:
SMOKE_TEST = False

## Generate synthetic example data

Run the next cell to create synthetic customer and transaction tables.
`customers` has three targets: `CHURNED` (binary), `SEGMENT` (multiclass) and `CREDIT_LIMIT` (regression). The last 80 of 800 customers have all three set to null; these are the rows to predict.
You can skim the generation code; the Kumo workflow starts below.

In [4]:
rng = np.random.default_rng(0)
customers_frame = pd.DataFrame({
    "CUSTOMER_ID": range(800),
    "AGE": rng.integers(20, 80, 800),
    "INCOME": rng.normal(60000, 15000, 800).round(),
})
customers_frame["CHURNED"] = pd.array(
    customers_frame.INCOME + rng.normal(0, 8000, 800) < 55000, dtype="boolean"
)
customers_frame["SEGMENT"] = np.select(
    [customers_frame.INCOME > 70000, customers_frame.AGE > 55], ["premium", "senior"], default="standard"
)
customers_frame["CREDIT_LIMIT"] = (customers_frame.INCOME * 0.2 + rng.normal(0, 500, 800)).round()

unlabeled_ids = np.arange(720, 800)
customers_frame.loc[unlabeled_ids, ["CHURNED", "SEGMENT", "CREDIT_LIMIT"]] = None
customers_frame.to_parquet(source_dir / "customers.parquet", index=False)

pd.DataFrame({
    "TX_ID": range(6000),
    "CUSTOMER_ID": rng.integers(0, 800, 6000),
    "TS": pd.Timestamp("2024-01-01") + pd.to_timedelta(rng.integers(0, 180, 6000), unit="D"),
    "AMOUNT": rng.gamma(2.0, 40.0, 6000).round(2),
}).to_parquet(source_dir / "transactions.parquet", index=False)

## Build the graph

`customers` has no time column, so queries on it are static. `transactions` has a timestamp and points to its customer.
See [Graphs](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs).

In [5]:
connector = kumo.FileConnector(root_dir=str(source_dir))
customers = kumo.Table.from_source_table(
    connector["customers.parquet"], primary_key="CUSTOMER_ID"
).infer_metadata()
transactions = kumo.Table.from_source_table(
    connector["transactions.parquet"], primary_key="TX_ID", time_column="TS"
).infer_metadata().with_column("CUSTOMER_ID", stype=Stype.ID)
graph = kumo.Graph(
    tables={"customers": customers, "transactions": transactions},
    edges=[kumo.Edge(src_table="transactions", fkey="CUSTOMER_ID", dst_table="customers")],
)
graph.validate()
snapshot = graph.snapshot()

## Write three static queries

Kumo infers the task from the target column: a comparison is binary, a string column is multiclass and a numeric column is regression.
Print each task type and expect `binary_classification`, `multiclass_classification` and `regression`.

In [6]:
binary_query = kumo.PredictiveQuery(
    graph, "PREDICT customers.CHURNED = True FOR EACH customers.CUSTOMER_ID"
)
multiclass_query = kumo.PredictiveQuery(
    graph, "PREDICT customers.SEGMENT FOR EACH customers.CUSTOMER_ID"
)
regression_query = kumo.PredictiveQuery(
    graph, "PREDICT customers.CREDIT_LIMIT FOR EACH customers.CUSTOMER_ID"
)
for query in (binary_query, multiclass_query, regression_query):
    print(query.get_task_type())

binary_classification
multiclass_classification
regression


## Train and predict

The helper trains one model per query with the FAST plan and scores the default population. It uses one experiment without refitting, and explanations are disabled, to keep this module on prediction output.
Binary prediction takes an explicit decision threshold. See [Training](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/training).

In [7]:
def fast_plan(query, snapshot, **suggest_options):
    model_plan = query.suggest_model_plan(snapshot, run_mode=RunMode.FAST, **suggest_options)
    model_plan.training_job.num_experiments = 1
    model_plan.training_job.refit_trainval = False
    model_plan.training_job.refit_full = False
    model_plan.training_job.disable_explain = True
    if SMOKE_TEST:
        model_plan.optimization.max_epochs = 2
        model_plan.optimization.max_steps_per_epoch = 20
    return model_plan


def train_and_predict(query, **predict_options):
    trainer = kumo.Trainer(model_plan=fast_plan(query, snapshot))
    trainer.fit(query.generate_training_table(snapshot), snapshot=snapshot)
    prediction_table = query.generate_prediction_table(snapshot)
    frame = trainer.predict(prediction_table, snapshot=snapshot, **predict_options).frame
    return trainer, frame


binary_trainer, binary = train_and_predict(binary_query, binary_classification_threshold=0.5)
_, multiclass = train_and_predict(multiclass_query)
_, regression = train_and_predict(regression_query)

## Read the output layout

Binary and regression return one row per entity. Binary adds a probability for each class; regression returns one predicted value.

In [8]:
binary.head(3)

,ENTITY,TARGET_PRED,False_PROB,True_PROB
0,775,False,0.732974,0.267026
1,778,False,0.753159,0.246841
2,781,True,0.236822,0.763178


In [9]:
regression.head(3)

,ENTITY,TARGET_PRED
0,720,11273.390625
1,723,9124.274414
2,726,10777.477539


Multiclass returns one row per entity and class, with `PREDICTED` marking the winning class. Pivot on `CLASS` for one row per entity; the three scores sum to about one.

In [10]:
multiclass.pivot(index="ENTITY", columns="CLASS", values="SCORE").head(3)

CLASS,premium,senior,standard
ENTITY,,,
720,0.0,1.0,0.0
721,0.0,1.0,0.0
722,0.0,0.0,1.0


## Choose the population

By default a static query scores the entities whose target is null, which here are exactly the 80 unlabeled customers.

In [11]:
print("Scored exactly the unlabeled customers:", set(binary.ENTITY) == set(unlabeled_ids))

Scored exactly the unlabeled customers: True


To score other rows, including labeled ones, write a table with an `ENTITY1` column and build the prediction table from it.
Pass the trained model's `training_table_generation_plan` so the table matches the fitted model. Expect one row per requested customer.

In [12]:
requested = [3, 17, 42, 500, 799]
pd.DataFrame({"ENTITY1": requested}).to_parquet(source_dir / "population.parquet", index=False)

supplied = kumo.PredictionTable.from_source_table(
    connector["population.parquet"],
    pq=binary_query,
    snapshot=snapshot,
    plan=binary_trainer.training_table_generation_plan,
)
binary_trainer.predict(supplied, snapshot=snapshot, binary_classification_threshold=0.5).frame

,ENTITY,TARGET_PRED,False_PROB,True_PROB
0,3,False,0.911409,0.088591
1,500,True,0.097269,0.902731
2,17,False,0.888870,0.111130
3,799,True,0.258307,0.741693
4,42,False,0.770839,0.229161


## Two rules that surprise people

**A time column makes a static query temporal.** If the entity table has a time column, a supplied population must also carry a timezone-naive `TIMESTAMP` column.
Add a signup time to `customers` and build a second graph from it.

In [13]:
customers_frame.assign(
    SIGNUP_TS=pd.Timestamp("2024-01-01") + pd.to_timedelta(rng.integers(0, 150, 800), unit="D")
).to_parquet(source_dir / "customers_timed.parquet", index=False)

timed_customers = kumo.Table.from_source_table(
    connector["customers_timed.parquet"], primary_key="CUSTOMER_ID", time_column="SIGNUP_TS"
).infer_metadata()
timed_graph = kumo.Graph(tables={"customers": timed_customers}, edges=[])
timed_snapshot = timed_graph.snapshot()
timed_query = kumo.PredictiveQuery(
    timed_graph, "PREDICT customers.CHURNED = True FOR EACH customers.CUSTOMER_ID"
)

Try the same population without `TIMESTAMP`. Expect an error that names the missing column.

In [14]:
pd.DataFrame({"ENTITY1": requested}).to_parquet(source_dir / "without_timestamp.parquet", index=False)
try:
    kumo.PredictionTable.from_source_table(
        connector["without_timestamp.parquet"], pq=timed_query, snapshot=timed_snapshot
    )
except ValueError as error:
    print(error)

Required column not found. Source table is missing TIMESTAMP, which is the name DiskGraph reads that column under and the only name it reads it under. It carries ['ENTITY1'].


Add a `TIMESTAMP` column. Expect the five rows to be accepted.

In [15]:
pd.DataFrame({"ENTITY1": requested, "TIMESTAMP": pd.Timestamp("2024-06-01")}
             ).to_parquet(source_dir / "with_timestamp.parquet", index=False)
with_timestamp = kumo.PredictionTable.from_source_table(
    connector["with_timestamp.parquet"], pq=timed_query, snapshot=timed_snapshot
)
print("Accepted rows:", with_timestamp.row_count)

Accepted rows: 5


**`prediction_time` is rejected for a static query**, which has no time window to anchor.

In [16]:
try:
    binary_query.generate_prediction_table(
        snapshot, prediction_time=pd.Timestamp("2024-06-01").to_pydatetime()
    )
except ValueError as error:
    print(error)

prediction_time is not supported for static queries


## Next step

Write a static query on your entity table, let Kumo score the rows with a null target, or supply an `ENTITY1` table when you need specific rows.
To train on rows you label yourself, see [temporal custom training table](local_kumo_engine_temporal_custom_training_table.ipynb); for temporal populations, see [custom prediction table](local_kumo_engine_custom_prediction_table.ipynb).

## Cleanup

Remove this notebook's workspace, which holds only the files it created. Skip this cell to keep the generated tables and predictions.

In [17]:
import shutil

shutil.rmtree(workspace)